In [1]:
! pip install azure-cosmos

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [azure-cosmos] [azure-cosmos]


In [5]:
from azure.cosmos import CosmosClient, PartitionKey
import uuid

endPoint = ""
key = ""

client = CosmosClient(endPoint, credential=key)

ServiceResponseError: Invalid URL '': No scheme supplied. Perhaps you meant https://?

In [4]:
#Container

db_name = 'analytics'
container_name = 'ventas'
partition_key_path = '/clientId'

database = client.create_database_if_not_exists(id = db_name)
container = database.create_container_if_not_exists(
    id = container_name,
    partition_key=PartitionKey(path=partition_key_path),
    offer_throughput=400
)



NameError: name 'client' is not defined

In [ ]:
#ingresar un elemento
item = {
    'id': 'C-001',
    'fecha': '2026-06-22',
    'monto': 1500,
    'detalle': {'canal':'online', 'moneda':'dol'}
}

container.upsert_item(item)

In [12]:
import random
Codigos = [f'C.{random.randint(1,100)}' for _ in range(10)]

In [16]:
import pandas as pd

df = pd.DataFrame({
    'clientId': ['C-001','C-002','C-003'],
    'fecha': ['2026-05-10','2026-05-30','2026-06-22'],
    'monto': [1500, 2200, 1700]
})

In [17]:
docs = []
for _, r in df.iterrows():
    docs.append({
        'id':str(uuid.uuid4()),
        'clientID':str(r['clientId']),
        'fecha':str(r['fecha']),
        'monto':float(r['monto'])
    })

In [18]:
docs

[{'id': '510acfc3-27d9-4b05-a7b2-a8e64d114409',
  'clientID': 'C-001',
  'fecha': '2026-05-10',
  'monto': 1500.0},
 {'id': 'cc1ef9c1-8383-4326-b0b2-7bf8da5edad3',
  'clientID': 'C-002',
  'fecha': '2026-05-30',
  'monto': 2200.0},
 {'id': 'd6aa89f7-9daa-4618-8a2c-b37f486984d6',
  'clientID': 'C-003',
  'fecha': '2026-06-22',
  'monto': 1700.0}]

In [ ]:
#escribir en paralelo
from concurrent.futures import ThreadPoolExecutor

def _upsert(docs):
    container.upsert_item(docs)

with ThreadPoolExecutor(max_workers=32) as ex:
    ex.map(_upsert, docs)

In [ ]:
#LEer Datos
query = 'select * from c'

#recuperar todos los items (con paginacion interna)
items = list(container.query_items(
    query=query,
    enable_cross_partition_query=True
))

#mostrar algunos resultados
for item in items[:5]:
    print(item)

In [ ]:
#LEer Datos con parametros
query = 'select c.id, c.clientId, c.monto from c where c.clientId=@cid'
#recuperar todos los items (con paginacion interna)
items = list(container.query_items(
    query=query,
    parameters=[{'name':'@cid', 'value': 'C-001'}],
    enable_cross_partition_query=True
))

#mostrar algunos resultados
print(items)